# Diagnosing High RAM Usage in Linux

In this notebook, we will deliberately create a controlled process that consumes increasing amounts of RAM and then diagnose it using standard Linux troubleshooting tools.

The exercise simulates a production scenario where an AI/ML application becomes slow because memory usage keeps increasing.

### What you will learn

- Inspect system specifications
- Check available RAM and swap
- Create a controlled memory-consuming process
- Identify the process consuming the most RAM
- Inspect process memory statistics
- Monitor memory growth over time
- Investigate the root cause
- Stop the problematic process safely
- Understand how the same workflow applies to AI/ML systems


# 1. The Production Scenario

Imagine you are an AI/ML engineer responsible for a Linux server.

A monitoring system reports:

> **The production server is becoming slow and application requests are starting to fail.**

You do not initially know the cause.

Our goal is to reproduce this problem in a controlled environment:

```text
Linux Server
│
├── Normal Linux Processes
├── Docker Services
├── AI/ML Application
│
└── memory_hog.py  ← Simulated Problem
        │
        └── Continuously consumes RAM
```

The process below is intentionally designed to retain allocated memory.

> ⚠️ **Safety note:** Do not let this process consume all available memory. Stop it after observing the behavior.


# 2. Identify System Specifications

Before diagnosing a performance problem, first understand the machine.

We need to know:

- CPU capacity
- Available RAM
- Swap configuration
- Disk capacity
- GPU resources, if applicable

This information helps determine whether high memory usage is normal for the workload or indicates a problem.


## 2.1 Check CPU Information

Run:

```bash
lscpu
```

Important information includes:

- Number of CPUs / cores
- CPU architecture
- CPU model
- Threads per core


In [ ]:
!lscpu | head -20

## 2.2 Check RAM and Swap

Run:

```bash
free -h
```

Important columns:

- **total** — Total installed memory
- **used** — Memory currently in use
- **free** — Completely unused memory
- **available** — Memory estimated to be available for new applications

Swap usage is also important. Heavy swap usage may indicate memory pressure.


In [ ]:
!free -h

## 2.3 Check Disk Space

Run:

```bash
df -h
```

This checks filesystem usage.

Disk pressure can also affect application performance, especially when swap or temporary files are heavily used.


In [ ]:
!df -h

## 2.4 Check GPU Resources (Optional)

If the machine has an NVIDIA GPU:

```bash
nvidia-smi
```

For AI/ML workloads, remember that **system RAM and GPU VRAM are different resources**.

A model may have:

```text
High CPU RAM usage
AND/OR
High GPU VRAM usage
```


In [ ]:
!command -v nvidia-smi >/dev/null && nvidia-smi || echo "nvidia-smi is not available on this system." 

# 3. Create a Controlled RAM-Consumption Problem

Now we intentionally create a process that continuously allocates memory.

This simulates problems such as:

- Memory leaks
- Unlimited caching
- Accumulating documents
- Retained embeddings
- Unbounded request history
- Repeatedly storing large objects


## 3.1 Create `memory_hog.py`

The following program allocates memory gradually.

Instead of immediately consuming all RAM, it allocates approximately **50 MB every 2 seconds**.

The limit can be changed using `MAX_MB`.

This makes the exercise safer than an unlimited allocation loop.


In [ ]:
%%writefile memory_hog.py
import time
import os

CHUNK_MB = 50
MAX_MB = 1000  # Stop automatically after approximately 1 GB

chunks = []

print(f"PID: {os.getpid()}")
print("Starting controlled RAM allocation...")

while len(chunks) * CHUNK_MB < MAX_MB:
    chunks.append(bytearray(CHUNK_MB * 1024 * 1024))

    allocated_mb = len(chunks) * CHUNK_MB
    print(f"Allocated approximately {allocated_mb} MB")

    time.sleep(2)

print("Reached configured memory limit.")
print("Keeping memory allocated for 60 seconds so it can be inspected.")

time.sleep(60)

print("Exiting normally.")


## How the Program Works

```text
Start
 ↓
Allocate 50 MB
 ↓
Store reference in `chunks`
 ↓
Wait 2 seconds
 ↓
Allocate another 50 MB
 ↓
Store reference
 ↓
Repeat
```

Because references are stored inside the `chunks` list, Python cannot release those allocated objects while the list still exists.

This simulates **unbounded memory retention**.


# 4. Start the Problem Process

Run the process in the background:

```bash
python3 memory_hog.py > memory_hog.log 2>&1 &
```

Then capture its PID.

The PID is important because Linux troubleshooting tools use it to inspect the process.


In [ ]:
!python3 memory_hog.py > memory_hog.log 2>&1 & echo $! > memory_hog.pid
!cat memory_hog.pid

## Inspect the Process Output

The log should show memory increasing:

```text
Allocated approximately 50 MB
Allocated approximately 100 MB
Allocated approximately 150 MB
...
```


In [ ]:
!cat memory_hog.log 2>/dev/null || true

# 5. Diagnose the Problem as if You Do Not Know the Cause

Now imagine that you did **not** write `memory_hog.py`.

You only know that:

> The Linux server is becoming slow and available memory is decreasing.

We will now diagnose the issue step by step.


# Step 1 — Confirm Memory Pressure

Run:

```bash
free -h
```

Look for:

- High used memory
- Low available memory
- Increasing swap usage

Example:

```text
Mem:   16Gi   14Gi   700Mi   1.2Gi
Swap:   4Gi   2.5Gi   1.5Gi
```

Possible interpretation:

```text
RAM is under pressure
        ↓
Swap may be used
        ↓
System performance can degrade
```


In [ ]:
!free -h

# Step 2 — Find the Largest Memory-Consuming Process

Run:

```bash
ps aux --sort=-%mem | head -10
```

This sorts processes by memory usage.

Important columns:

- PID
- %MEM
- COMMAND

Our goal is to identify the suspicious process.


In [ ]:
!ps aux --sort=-%mem | head -10

# Step 3 — Identify the PID

Read the PID created earlier:

```bash
cat memory_hog.pid
```

In a real incident, you may discover the PID from:

```bash
ps aux --sort=-%mem
```


In [ ]:
!cat memory_hog.pid

# Step 4 — Inspect Detailed Process Memory Information

Run:

```bash
ps -p PID -o pid,ppid,%mem,rss,vsz,etime,cmd
```

Replace `PID` with the process ID.

Important values:

### RSS

**RSS (Resident Set Size)** represents memory currently resident in physical RAM.

### VSZ

**VSZ (Virtual Memory Size)** represents the virtual memory space available to the process.

### ELAPSED

Shows how long the process has been running.


In [ ]:
!pid=$(cat memory_hog.pid)
!ps -p "$pid" -o pid,ppid,%mem,rss,vsz,etime,cmd

# Step 5 — Monitor Memory Growth

A single memory measurement is not enough.

We need to answer:

> Is memory stable, or is it continuously increasing?

Run this command in a terminal:

```bash
watch -n 2 "ps -p PID -o pid,%mem,rss,vsz,etime,cmd"
```

Example progression:

```text
Time 1 → RSS = 500 MB
Time 2 → RSS = 700 MB
Time 3 → RSS = 900 MB
Time 4 → RSS = 1.0 GB
```

If memory continuously grows, possible causes include:

```text
Memory leak
Unlimited cache
Growing data structures
Repeated model loading
Large documents retained in memory
Unbounded request history
```


In [ ]:
!watch -n 2 "ps -p PID -o pid,%mem,rss,vsz,etime,cmd"

# Step 6 — Inspect With `top` or `htop`

For a live terminal view:

```bash
top -p PID
```

Example:

```text
PID      USER    %CPU   %MEM   RES    COMMAND
12345    user     1.0    15.0   2G     python3
```

You can also use:

```bash
htop
```

In `htop`, sort processes by memory usage to quickly identify the largest consumer.


# Step 7 — Root Cause Analysis

The diagnostic path now looks like this:

```text
High RAM Usage
      ↓
Check System Memory
      ↓
Identify Largest Process
      ↓
Inspect Process Details
      ↓
Monitor Memory Over Time
      ↓
Memory Continuously Increases
      ↓
Investigate Application
      ↓
ROOT CAUSE
```

When we inspect the simulated application:

```python
chunks = []

while True:
    chunks.append(bytearray(...))
```

We discover the root cause.

The application:

```text
Allocates memory
      ↓
Stores references in a growing list
      ↓
Does not release those references
      ↓
Allocates more memory
      ↓
Repeats
```

### Root Cause

> **Unbounded memory allocation and retention.**

The process continuously retains objects, causing RAM usage to grow.


# Step 8 — Stop the Problematic Process

First try a normal termination:

```bash
kill PID
```

If the process does not terminate, inspect it before using stronger signals.

For this controlled exercise, we can stop the process and then verify that it has exited.


In [ ]:
!pid=$(cat memory_hog.pid)
!kill "$pid" 2>/dev/null || true
!sleep 2
!ps -p "$pid" || echo "Process has stopped." 

# Step 9 — Verify Memory Recovery

After stopping the process:

```bash
free -h
```

The system should eventually recover memory.

Remember that Linux uses available RAM for caching, so **used memory alone is not always the best indicator**.

Pay particular attention to:

```text
available
```


In [ ]:
!free -h

# 10. Complete Linux Diagnostic Workflow

```text
                    PROBLEM
                       │
                       ▼
              RAM usage is very high
                       │
                       ▼
              Check system resources
                       │
                       ▼
                   free -h
                       │
                       ▼
              Memory pressure detected
                       │
                       ▼
             Find largest process
                       │
                       ▼
          ps aux --sort=-%mem | head
                       │
                       ▼
             Process identified
                       │
                       ▼
             Inspect process details
                       │
                       ▼
          ps -p PID -o ...
                       │
                       ▼
          Monitor memory over time
                       │
                       ▼
             RSS continuously grows
                       │
                       ▼
              Investigate application
                       │
                       ▼
                Find root cause
                       │
                       ▼
                 Fix the issue
                       │
                       ▼
               Verify improvement
```


# 11. Why This Matters for an AI/ML Engineer

The dummy process is simple, but the diagnostic methodology is directly applicable to production AI/ML systems.

A real production process might be:

```text
uvicorn app.main:app
```

and internally:

```text
FastAPI
   ↓
RAG Pipeline
   ↓
Embedding Model
   ↓
Document Processing
   ↓
Vector Database
   ↓
LLM
```

The Linux-level investigation remains the same.


## Common AI/ML Root Causes of High RAM Usage

### 1. Model Loaded Repeatedly

```text
Request 1 → Load model
Request 2 → Load model
Request 3 → Load model
```

Better:

```text
Application Startup
        ↓
Load Model Once
        ↓
Reuse Model
```

---

### 2. Entire Dataset Loaded Into Memory

```text
1,000,000 documents
        ↓
Load all into RAM
        ↓
Memory exhausted
```

Better:

```text
Batch 1 → Process → Store
Batch 2 → Process → Store
Batch 3 → Process → Store
```

---

### 3. Unlimited Cache

```text
Request
   ↓
Store in cache
   ↓
Never remove
```

Over time, memory continuously grows.

---

### 4. Memory Leak

Objects that are no longer useful remain referenced and cannot be released.

---

### 5. Excessive Concurrency

Too many simultaneous requests may cause multiple copies of:

- Documents
- Embeddings
- Model inputs
- Intermediate tensors

to exist in memory simultaneously.


# 12. Practical AI/ML Example

Suppose an AI engineer receives this alert:

> **RAG API latency increased from 2 seconds to 20 seconds.**

The engineer investigates:

```text
Latency Increased
       ↓
Check CPU / RAM
       ↓
RAM nearly exhausted
       ↓
Find largest process
       ↓
uvicorn consuming 80% of RAM
       ↓
Monitor RSS
       ↓
RSS continuously increases
       ↓
Inspect application
       ↓
Document chunks stored indefinitely
       ↓
ROOT CAUSE FOUND
```

Possible solution:

```text
Batch documents
       +
Limit cache
       +
Store persistent data externally
       +
Control concurrency
       +
Reuse loaded models
```


# 13. Final Summary

## The Problem

A Linux process continuously consumes RAM.

## The Diagnostic Approach

```text
Observe
  ↓
Measure
  ↓
Identify process
  ↓
Inspect process
  ↓
Monitor memory growth
  ↓
Investigate application
  ↓
Find root cause
  ↓
Fix
  ↓
Verify
```

## Why This Matters for AI/ML Engineering

AI/ML engineers work with memory-intensive systems involving:

- Large language models
- Embedding models
- RAG pipelines
- Large datasets
- Vector databases
- Document processing
- Model inference APIs

The same Linux troubleshooting skills help engineers:

- Prevent out-of-memory crashes
- Improve application reliability
- Reduce latency
- Support more users
- Optimize infrastructure costs
- Build scalable production AI systems

> **Key takeaway:** Building an AI model is only part of AI/ML engineering. In production, engineers must also diagnose and manage CPU, RAM, GPU, storage, processes, and system reliability.
